# HORN tutorial: from a single oscillator to sequential MNIST

A hands-on introduction to **Harmonic Oscillator Recurrent Networks (HORNs)**, after

> F. Effenberger, P. Carvalho, I. Dubinin & W. Singer, *The functional role of oscillatory dynamics in neocortical circuits: A computational perspective*, PNAS 122(4) e2412830122 (2025). Reference code: [github.com/exilef/horn](https://github.com/exilef/horn).

Everything here runs on a small, pure-NumPy implementation that lives in the `horn/` package of this repository. There is no autograd: the backward pass (backpropagation through time) is written out by hand in `horn/network.py`, and we will verify it numerically. The idea is that you can read every line of code that is executed.

**Roadmap**

| Part | What you will learn |
|---|---|
| 1 | A single node: the damped harmonic oscillator (DHO), its parameters $\omega, \gamma, \alpha$, resonance, phase coding, feedback |
| 2 | Coupling nodes into a recurrent network: the HORN equations, synchrony, heterogeneity |
| 3 | Learning I: loss functions and gradient descent on the readout only ("reservoir computing") |
| 4 | Learning II: backpropagation through time (BPTT), gradient checking, gradient flow |
| 5 | Training a tiny HORN on XOR (classic and temporal) |
| 6 | Sequential MNIST: resonance with pixel statistics, training, and looking inside the trained network |

Cells marked **Try it** are invitations to change parameters. Many figures come with sliders (via `ipywidgets`); if widgets do not render in your environment the same functions are simply called once with default values.

In [ ]:
import os, sys, time, inspect
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

try:                                              # Jupyter: draw figures under the cell
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

# make the `horn` package importable when the notebook runs from HORN/notebooks/
REPO = Path.cwd().resolve()
if not (REPO / 'horn').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from horn import *          # HORN, ElmanRNN, simulate_dho, train, ... (see horn/__init__.py)

plt.rcParams.update({'figure.figsize': (9, 3.2), 'figure.dpi': 100, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.3})

# --- interactivity helper ------------------------------------------------------
# `interactive(fn, a=slider, ...)` shows sliders when ipywidgets is available and
# otherwise calls fn once with the slider defaults (also used when HORN_STATIC=1).
STATIC = os.environ.get('HORN_STATIC') == '1'
try:
    import ipywidgets as W
    from ipywidgets import interact
except ImportError:
    W, STATIC = None, True

if W is None:
    from types import SimpleNamespace
    class _Ctl:
        def __init__(self, value=None, **kw): self.value = value
    W = SimpleNamespace(FloatSlider=_Ctl, IntSlider=_Ctl, FloatLogSlider=_Ctl, Checkbox=_Ctl, Dropdown=_Ctl)

def interactive(fn, **controls):
    if STATIC:
        return fn(**{k: c.value for k, c in controls.items()})
    for c in controls.values():
        if hasattr(c, 'continuous_update'):
            c.continuous_update = False
    return interact(fn, **controls)

OMEGA, GAMMA, ALPHA = 2 * np.pi / 28, 0.01, 0.04     # the paper's default node parameters for sMNIST
def xor_bits(i):
    return tuple(int(b) for b in XOR_TABLE[i])
print(f'omega = {OMEGA:.4f} rad/step (period {2*np.pi/OMEGA:.0f} steps), gamma = {GAMMA}, alpha = {ALPHA}')

## 1. One node: the damped harmonic oscillator

In a HORN every node is a **driven, damped harmonic oscillator** (DHO). Think of a mass on a spring in a viscous fluid, or, biologically, of the aggregate activity of a small circuit of excitatory and inhibitory neurons that tends to oscillate (a cortical column, a PING circuit). A node has a single state variable, the amplitude $x$, plus its velocity $y = \dot x$, and three *control parameters*:

| symbol | name | meaning |
|---|---|---|
| $\omega$ | natural frequency | the node wants to oscillate with period $2\pi/\omega$ |
| $\gamma$ | damping | oscillations decay like $e^{-\gamma t}$; $1/\gamma$ is the node's memory time |
| $\alpha$ | excitability | how strongly input kicks the velocity |

The continuous-time equation is $\ddot x + 2\gamma \dot x + \omega^2 x = \alpha \tanh(I(t))$, and the network is simulated in discrete time with the *symplectic Euler* scheme (step size $h=1$):

$$
\begin{aligned}
y_{t+1} &= y_t + \alpha\,\tanh(u_t) - 2\gamma\, y_t - \omega^2 x_t \\
x_{t+1} &= x_t + y_{t+1}
\end{aligned}
$$

where $u_t$ is the total drive (input, and later also recurrent input). Two things are worth noticing already:

* The input enters through a **tanh**: the node has an input nonlinearity (saturation), but its own dynamics are linear.
* The input acts on the **velocity**, not directly on the amplitude ("coupling on the velocity term" in the paper).

Let's kick a node once and watch it ring.

In [ ]:
T = 300
s = impulse(T, t_on=20, width=1, amplitude=5.0)      # a single kick at t = 20
x, y = simulate_dho(s, OMEGA, GAMMA, ALPHA)           # x, y have length T+1 (index 0 = initial state)
t = np.arange(T + 1)

fig, ax = plt.subplots(2, 1, figsize=(9, 4.8), sharex=True, gridspec_kw=dict(height_ratios=[1, 2.5]))
ax[0].plot(t[1:], s, 'k'); ax[0].set_ylabel('input s(t)')
ax[1].plot(t, x, label='amplitude x'); ax[1].plot(t, y, alpha=0.6, label='velocity y')
A = np.abs(x).max()
ax[1].plot(t[21:], A * np.exp(-GAMMA * (t[21:] - 21)), 'k--', lw=0.8, label=r'envelope $\propto e^{-\gamma t}$')
for k in range(1, 10):                                 # one marker per natural period
    ax[1].axvline(21 + k * 2 * np.pi / OMEGA, color='gray', lw=0.5, ls=':')
ax[1].legend(ncol=3); ax[1].set_xlabel('time step'); ax[1].set_ylabel('state')
plt.show()

info = eigen_analysis(OMEGA, GAMMA)
print(f"regime: {info['regime']}, amplitude decay per step: {info['decay']:.4f}, period: {info['period']:.1f} steps")

The kick changes the velocity by $\alpha\tanh(5)\approx\alpha$, after which the node oscillates freely with period $2\pi/\omega = 28$ steps and an amplitude that decays by a factor $\approx 1-\gamma$ per step. With $\gamma = 0.01$ the memory of a single input lasts for a few hundred steps, i.e. many oscillation periods.

**Try it:** move the sliders. What happens when $\gamma$ approaches $\omega$ (critical damping), or exceeds it? The paper's central comparison is between networks of nodes in the *oscillatory* regime ($\gamma < \omega$) and the same networks with *non-oscillatory*, integrator-like nodes ($\gamma > \omega$): everything else stays identical, only the nodes stop ringing.

In [ ]:
def show_node(omega=OMEGA, gamma=GAMMA, alpha=ALPHA):
    s = impulse(300, 20, 1, 5.0)
    x, _ = simulate_dho(s, omega, gamma, alpha)
    info = eigen_analysis(omega, gamma)
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(x); ax.axhline(0, color='k', lw=0.5)
    ax.set_xlabel('time step'); ax.set_ylabel('amplitude x')
    per = f"{info['period']:.1f} steps" if np.isfinite(info['period']) else 'none'
    ax.set_title(f"{info['regime']}   |   decay/step {info['decay']:.3f}   |   period {per}   |   critical damping at gamma = omega = {omega:.3f}")
    plt.show()

interactive(show_node,
            omega=W.FloatSlider(value=OMEGA, min=0.02, max=1.0, step=0.01, description='omega'),
            gamma=W.FloatLogSlider(value=GAMMA, base=10, min=-3, max=0, step=0.05, description='gamma'),
            alpha=W.FloatSlider(value=ALPHA, min=0.01, max=1.0, step=0.01, description='alpha'))

In [ ]:
# the same comparison as a static figure: from weakly damped to overdamped
fig, axes = plt.subplots(1, 4, figsize=(12, 2.6), sharey=True)
for ax, g in zip(axes, [0.002, 0.03, OMEGA, 0.5]):
    x, _ = simulate_dho(impulse(200, 10, 1, 5.0), OMEGA, g, ALPHA)
    ax.plot(x); ax.set_title(f'gamma = {g:.3f} ({eigen_analysis(OMEGA, g)["regime"]})', fontsize=9)
    ax.set_xlabel('time step')
axes[0].set_ylabel('x'); plt.tight_layout(); plt.show()

### 1.1 Any input becomes an oscillation

A leaky integrator smooths its input. A DHO *converts* its input into oscillations, even when the input is constant or noisy. For a constant drive $d$ the amplitude settles (with overshoot and ringing) at the fixed point $x^* = \alpha\tanh(d)/\omega^2$.

In [ ]:
T = 400
s_step = step_input(T, t_on=50, amplitude=1.0)
rng = np.random.default_rng(0)
s_noise = 1.0 * rng.standard_normal(T)

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
x, _ = simulate_dho(s_step, OMEGA, GAMMA, ALPHA)
ax[0].plot(s_step, 'k', alpha=0.4, label='input (step)'); ax[0].plot(x, label='x')
ax[0].axhline(ALPHA * np.tanh(1.0) / OMEGA**2, color='C3', ls='--', lw=0.8, label=r'fixed point $\alpha\tanh(d)/\omega^2$')
ax[0].legend(); ax[0].set_title('step input'); ax[0].set_xlabel('time step')
x, _ = simulate_dho(s_noise, OMEGA, GAMMA, ALPHA)
ax[1].plot(s_noise, 'k', alpha=0.25, label='input (white noise)'); ax[1].plot(x, label='x')
ax[1].legend(); ax[1].set_title('noise input: the node resonates at its own frequency'); ax[1].set_xlabel('time step')
plt.show()

### 1.2 Phase encodes temporal order

Because the node keeps ringing, the *timing* of inputs is preserved in the **phase** of the oscillation. Two pulses in the order big-then-small versus small-then-big deliver the same total input, but the node ends up in a different state. A network can read this out: the oscillation phase is an extra coding dimension that a non-oscillating unit does not have.

In [ ]:
T, t1, t2 = 150, 20, 27          # pulses 7 steps (a quarter period) apart
s_AB = impulse(T, t1, 1, 3.0) + impulse(T, t2, 1, 1.0)     # A (big) then B (small)
s_BA = impulse(T, t1, 1, 1.0) + impulse(T, t2, 1, 3.0)     # B then A
x_AB, _ = simulate_dho(s_AB, OMEGA, GAMMA, ALPHA)
x_BA, _ = simulate_dho(s_BA, OMEGA, GAMMA, ALPHA)

fig, ax = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True, gridspec_kw=dict(height_ratios=[1, 2]))
ax[0].plot(s_AB, label='A then B'); ax[0].plot(s_BA, '--', label='B then A'); ax[0].legend(); ax[0].set_ylabel('input')
ax[1].plot(x_AB, label='A then B'); ax[1].plot(x_BA, '--', label='B then A'); ax[1].legend()
ax[1].set_ylabel('x'); ax[1].set_xlabel('time step'); ax[1].set_title('same pulses, different order -> different phase of the response')
plt.show()

### 1.3 Resonance: the node as a frequency filter

Driven by a sinusoid of frequency $\omega_i$, the node responds with a sinusoid whose amplitude depends strongly on $\omega_i$: inputs near the natural frequency $\omega$ are *amplified*, fast inputs are attenuated. The **gain curve** $G(\omega_i)$ = output amplitude / input amplitude is what the paper calls the node's *intrinsic receptive field* (IRF): the frequency band in which the node is selective. For a linear oscillator the gain is known analytically (`gain_linear`, exact for the discretised system); because of the tanh the real node's gain additionally depends on the input amplitude (`gain_simulated` measures it by brute force).

In [ ]:
freqs = np.linspace(0.02, 1.2, 150)
g_lin = gain_linear(freqs, OMEGA, GAMMA, ALPHA)
g_small = gain_simulated(freqs, OMEGA, GAMMA, ALPHA, amplitude=0.1, T=3000)
g_large = gain_simulated(freqs, OMEGA, GAMMA, ALPHA, amplitude=5.0, T=3000)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.semilogy(freqs, g_lin, 'k', label='linear theory (exact for the discretised node)')
ax.semilogy(freqs, g_small, 'o', ms=3, label='simulated, input amplitude 0.1')
ax.semilogy(freqs, g_large, 's', ms=3, label='simulated, input amplitude 5 (tanh saturates)')
ax.axvline(OMEGA, color='gray', ls=':', label=r'$\omega$'); ax.axhline(1, color='gray', lw=0.5)
ax.set_xlabel(r'input frequency $\omega_i$ (rad / step)'); ax.set_ylabel('gain G'); ax.legend(fontsize=8)
ax.set_title('intrinsic receptive field (gain curve) of one node')
plt.show()
print(f'peak gain (linear): {g_lin.max():.1f} at omega_i = {freqs[g_lin.argmax()]:.3f};  DC gain alpha/omega^2 = {ALPHA/OMEGA**2:.2f}')

In [ ]:
def show_resonance(omega_in=OMEGA, amplitude=0.5):
    T = 600
    s = sinusoid(T, omega_in, amplitude)
    x, _ = simulate_dho(s, OMEGA, GAMMA, ALPHA)
    fig, ax = plt.subplots(1, 2, figsize=(12, 3), gridspec_kw=dict(width_ratios=[2, 1]))
    ax[0].plot(s, 'k', alpha=0.4, label='input'); ax[0].plot(x, label='response x'); ax[0].legend(loc='upper left')
    ax[0].set_xlabel('time step'); ax[0].set_title(f'input frequency {omega_in:.3f} (period {2*np.pi/omega_in:.1f} steps)')
    ax[1].semilogy(freqs, g_lin, 'k'); ax[1].axvline(OMEGA, color='gray', ls=':')
    g = 0.5 * (x[T//2:].max() - x[T//2:].min()) / amplitude
    ax[1].plot([omega_in], [g], 'ro'); ax[1].set_xlabel(r'$\omega_i$'); ax[1].set_ylabel('gain'); ax[1].set_title(f'gain = {g:.2f}')
    plt.show()

interactive(show_resonance,
            omega_in=W.FloatSlider(value=0.12, min=0.02, max=1.0, step=0.005, description='omega_in'),
            amplitude=W.FloatSlider(value=0.5, min=0.05, max=5.0, step=0.05, description='amplitude'))

### 1.4 Feedback reshapes the receptive field

In a HORN each node also receives its *own* output back inside the tanh: an **amplitude feedback** $v\,x_t$ and a **velocity feedback** $w\,y_t$ (the diagonal of the recurrent weight matrix). These are learned during training. In the linear regime they simply shift the effective oscillator parameters

$$\omega_{\text{eff}}^2 = \omega^2 - \alpha v, \qquad \gamma_{\text{eff}} = \gamma - \alpha w / 2,$$

so learning $v$ re-tunes the node's preferred frequency and learning $w$ changes its damping. This is how the paper's *intrinsic receptive fields* become diverse during learning (its Fig. 2E). For $w > 2\gamma/\alpha$ the effective damping becomes negative and the node turns into a **self-sustained oscillator** whose amplitude is limited only by the tanh.

In [ ]:
def show_feedback(v=0.0, w=0.0):
    om_eff, ga_eff = effective_parameters(OMEGA, GAMMA, ALPHA, v, w)
    fig, ax = plt.subplots(1, 2, figsize=(12, 3))
    ax[0].semilogy(freqs, g_lin, 'k', alpha=0.4, label='no feedback')
    ax[0].semilogy(freqs, np.clip(gain_linear(freqs, OMEGA, GAMMA, ALPHA, v, w), 1e-3, 1e4), 'C3', label=f'v = {v:.2f}, w = {w:.2f}')
    ax[0].axvline(OMEGA, color='gray', ls=':'); ax[0].set_ylim(1e-2, 1e3); ax[0].legend(); ax[0].set_xlabel(r'$\omega_i$'); ax[0].set_ylabel('gain')
    ax[0].set_title(f'effective omega = {om_eff:.3f}, effective gamma = {ga_eff:.4f}' + ('  (negative: self-sustained!)' if ga_eff <= 0 else ''))
    x, _ = simulate_dho(impulse(600, 20, 1, 5.0), OMEGA, GAMMA, ALPHA, v=v, w=w)
    ax[1].plot(x); ax[1].set_xlabel('time step'); ax[1].set_ylabel('x'); ax[1].set_title('impulse response with feedback')
    plt.show()

interactive(show_feedback,
            v=W.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.05, description='v (ampl.)'),
            w=W.FloatSlider(value=0.3, min=-0.5, max=0.7, step=0.02, description='w (veloc.)'))

<details><summary><b>Aside: the one-step matrix and why ω must stay below 2</b> (click to expand)</summary>

Ignoring the input, one update step is a linear map of the state $(x, y)$:

$$\begin{pmatrix} x_{t+1} \\ y_{t+1} \end{pmatrix} = M \begin{pmatrix} x_t \\ y_t \end{pmatrix},\qquad
M = \begin{pmatrix} 1-\omega^2 & 1-2\gamma \\ -\omega^2 & 1-2\gamma \end{pmatrix}.$$

Its eigenvalues $\lambda$ have modulus $\sqrt{1-2\gamma}\approx 1-\gamma$ (the decay per step) and, in the oscillatory regime, a phase angle that equals the oscillation frequency per step. With $h=1$ the scheme is only stable for $\omega < 2$. We will meet $M$ again in Part 4: the *same* matrix (transposed) propagates error signals backwards in time, which is why gradients in a HORN neither explode nor vanish quickly.
</details>

In [ ]:
M = linear_map(OMEGA, GAMMA)
lam = np.linalg.eigvals(M)
print('M =\n', M.round(4))
print('eigenvalues:', lam.round(4), '  |lambda| =', np.abs(lam[0]).round(4), '  angle =', np.angle(lam[0]).round(4), ' (omega =', round(OMEGA, 4), ')')

gammas = np.logspace(-3, 0, 200)
ea = [eigen_analysis(OMEGA, g) for g in gammas]
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].semilogx(gammas, [e['decay'] for e in ea]); ax[0].axvline(OMEGA, color='gray', ls=':'); ax[0].set_ylabel('decay per step |lambda|'); ax[0].set_xlabel('gamma')
ax[1].semilogx(gammas, [e['freq'] for e in ea]); ax[1].axvline(OMEGA, color='gray', ls=':', label='gamma = omega'); ax[1].set_ylabel('oscillation frequency'); ax[1].set_xlabel('gamma'); ax[1].legend()
plt.show()

## 2. From nodes to a network

A HORN is a set of $n$ DHO nodes that drive each other through their velocities. With vectors $\mathbf x_t, \mathbf y_t \in \mathbb R^n$ and per-node parameter vectors $\boldsymbol\omega, \boldsymbol\gamma, \boldsymbol\alpha$ the update is

$$
\begin{aligned}
\mathbf u_t &= \tfrac{1}{\sqrt n}\big(W_{hh}\,\mathbf y_t + \mathbf b_{hh} + \mathbf v \odot \mathbf x_t\big) + W_{ih}\,\mathbf s_t + \mathbf b_{ih} \\
\mathbf y_{t+1} &= \mathbf y_t + \boldsymbol\alpha \odot \tanh(\mathbf u_t) - 2\boldsymbol\gamma \odot \mathbf y_t - \boldsymbol\omega^2 \odot \mathbf x_t \\
\mathbf x_{t+1} &= \mathbf x_t + \mathbf y_{t+1} \\
\text{logits} &= W_{out}\,\mathbf x_T + \mathbf b_{out} \qquad\text{(affine readout at the last time step)}
\end{aligned}
$$

* $W_{ih}$ ($n \times n_{in}$) maps the input time series $\mathbf s_t$ onto the nodes, $W_{hh}$ ($n\times n$) is the recurrent coupling (its diagonal is the velocity feedback $w$), $\mathbf v$ the amplitude feedback. The factor $1/\sqrt n$ keeps the recurrent drive independent of network size.
* The **trainable parameters** are $W_{ih}, \mathbf b_{ih}, W_{hh}, \mathbf b_{hh}, \mathbf v, W_{out}, \mathbf b_{out}$ -- that is all. $\omega, \gamma, \alpha$ are fixed hyperparameters ("innate" properties of the nodes). A HORN has about $n^2 + n\,(n_{in} + n_{out} + 4)$ parameters, roughly half of a coRNN and a quarter of an LSTM with the same number of units.
* A **homogeneous** HORN gives all nodes the same $\omega,\gamma,\alpha$; a **heterogeneous** one draws $\omega$ and $\gamma$ per node from a range (here uniformly in $[0.5, 2]\times$ the base value, as in the authors' code).

In [ ]:
net = HORN(n_in=1, n_hidden=16, n_out=10)          # homogeneous, default parameters omega, gamma, alpha
for k, p in net.params.items():
    print(f'{k:6s} shape {str(p.shape):10s} {"(trainable)" if k in net.trainable else ""}')
print('trainable parameters:', net.n_params)
plot_matrix(net.params['W_hh'], title='W_hh at initialisation (diagonal = velocity feedback w)'); plt.show()

Let's kick an untrained network with a single pulse. In a **homogeneous** network all nodes ring at the same frequency, so the network response is dominated by global synchrony (some nodes in phase, some in anti-phase, depending on the sign of their input weight). In a **heterogeneous** network the nodes drift apart and the response is already much richer before any learning has taken place. We quantify synchrony with the time-resolved **Kuramoto order parameter** $r(t) = |\frac1n\sum_k e^{i\phi_k(t)}|$ ($r=1$: all phases equal, $r\approx 0$: phases spread out).

In [ ]:
S = impulse(400, 20, 1, 5.0)[None, :, None]        # batch of 1 sequence, 400 steps, 1 input channel
nets = {'homogeneous HORN (untrained)': HORN(1, 16, 10, seed=1),
        'heterogeneous HORN (untrained)': HORN.heterogeneous(1, 16, 10, seed=1)}

fig, ax = plt.subplots(2, 2, figsize=(12, 5.5), sharex=True, gridspec_kw=dict(height_ratios=[2.2, 1]))
for j, (name, m) in enumerate(nets.items()):
    X = m.simulate(S)['X'][0]                       # (T+1, n) trajectories of all nodes
    plot_traces(X, ax=ax[0, j], title=name)
    ax[1, j].plot(np.arange(30, X.shape[0]), kuramoto_order(X)[30:]); ax[1, j].set_ylim(0, 1.05)
    ax[1, j].set_ylabel('Kuramoto r(t)'); ax[1, j].set_xlabel('time step')
plt.tight_layout(); plt.show()
print('natural periods of the heterogeneous nodes:', np.round(2*np.pi/nets['heterogeneous HORN (untrained)'].omega, 1))

In [ ]:
# sanity check: a HORN with a single node and hand-set weights is exactly the single DHO of Part 1
one = HORN(n_in=1, n_hidden=1, n_out=2)
one.params['W_ih'][:] = 1.0; one.params['b_ih'][:] = 0.0; one.params['b_hh'][:] = 0.0
one.params['W_hh'][:] = 0.3          # = velocity feedback w (1/sqrt(1) = 1)
one.params['v'][:] = -0.2            # = amplitude feedback v
s = impulse(200, 10, 2, 2.0)
x_net = one.simulate(s[None, :, None])['X'][0, :, 0]
x_dho, _ = simulate_dho(s, OMEGA, GAMMA, ALPHA, v=-0.2, w=0.3)
print('max |difference| between network and single-node simulation:', np.abs(x_net - x_dho).max())

**Try it:** scale the recurrent weights up and down and toggle heterogeneity. With weak coupling the nodes behave almost independently; strong coupling makes the network dynamics increasingly complex.

In [ ]:
def show_network(coupling=1.0, heterogeneous=False, seed=1):
    m = HORN.heterogeneous(1, 16, 10, seed=seed) if heterogeneous else HORN(1, 16, 10, seed=seed)
    m.params['W_hh'] *= coupling
    X = m.simulate(S)['X'][0]
    fig, ax = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True, gridspec_kw=dict(height_ratios=[2.2, 1]))
    plot_traces(X, ax=ax[0], title=f'{"heterogeneous" if heterogeneous else "homogeneous"} HORN, W_hh scaled by {coupling:g}')
    ax[1].plot(np.arange(30, X.shape[0]), kuramoto_order(X)[30:]); ax[1].set_ylim(0, 1.05); ax[1].set_ylabel('r(t)'); ax[1].set_xlabel('time step')
    plt.show()

interactive(show_network, coupling=W.FloatLogSlider(value=1.0, base=10, min=-1, max=2, step=0.1, description='coupling'),
            heterogeneous=W.Checkbox(value=False, description='heterogeneous'), seed=W.IntSlider(value=1, min=0, max=10, description='seed'))

## 3. Learning I: the readout and gradient descent

Time to make the network *do* something. The task format throughout this notebook is **sequence classification**: a time series goes in, and at the last time step we read out a class. The readout is an affine map followed by a softmax:

$$\text{logits} = W_{out}\mathbf x_T + \mathbf b_{out},\qquad p_c = \frac{e^{\text{logit}_c}}{\sum_{c'} e^{\text{logit}_{c'}}},\qquad L = -\log p_{\text{label}}$$

$L$ is the **cross-entropy loss** (averaged over a mini-batch). *Learning* means changing the parameters so that $L$ goes down, and the workhorse for that is **gradient descent**: compute $\partial L/\partial\theta$ for every parameter $\theta$ and take a small step against it, $\theta \leftarrow \theta - \eta\,\partial L/\partial\theta$.

For the readout the gradient is easy and worth remembering. Softmax and cross-entropy combine to

$$\frac{\partial L}{\partial \text{logits}} = \mathbf p - \text{onehot}(\text{label})$$

(*predicted minus target*), and because the logits are linear in $W_{out}$,

$$\frac{\partial L}{\partial W_{out}} = (\mathbf p - \text{onehot})\,\mathbf x_T^{\top},\qquad \frac{\partial L}{\partial \mathbf b_{out}} = \mathbf p - \text{onehot}.$$

This is exactly the output layer of a multilayer perceptron. If we train **only the readout** and leave the recurrent network at its random initialisation, we are doing *reservoir computing*: the untrained oscillator network acts as a fixed, nonlinear feature extractor. Let's try that on a toy task that plays to the strength of oscillators: telling two frequencies apart in heavy noise.

In [ ]:
FREQS = (OMEGA, 3 * OMEGA)                    # class 0: at the nodes' resonance, class 1: three times faster
S_f, y_f, clean_f = make_frequency_task(n_samples=300, T=150, freqs=FREQS, noise=1.0, seed=0)
S_f_test, y_f_test, _ = make_frequency_task(n_samples=300, T=150, freqs=FREQS, noise=1.0, seed=1)

fig, ax = plt.subplots(1, 2, figsize=(12, 2.8), sharey=True)
for c in (0, 1):
    i = np.where(y_f == c)[0][0]
    ax[c].plot(S_f[i, :, 0], 'k', alpha=0.4, label='noisy input (what the network sees)')
    ax[c].plot(clean_f[i, :, 0], 'C3', label='underlying sinusoid')
    ax[c].set_title(f'class {c}: frequency {FREQS[c]:.3f}'); ax[c].legend(fontsize=8); ax[c].set_xlabel('time step')
plt.show()

In [ ]:
reservoir = HORN.heterogeneous(n_in=1, n_hidden=8, n_out=2, seed=0)     # random, untrained
out = reservoir.simulate(S_f)
x_T = out['X'][:, -1, :]                                                # final states, shape (N, n)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
for c, col in zip((0, 1), ('C0', 'C1')):
    i = np.where(y_f == c)[0][0]
    plot_traces(out['X'][i], ax=ax[0], alpha=0.6, lw=0.8, cmap='Blues' if c == 0 else 'Oranges')
ax[0].set_title('node trajectories for one example of each class (blue: class 0, orange: class 1)')
ax[1].scatter(x_T[:, 0], x_T[:, 1], c=y_f, cmap='coolwarm', s=12)
ax[1].set_xlabel('x_T of node 0'); ax[1].set_ylabel('x_T of node 1'); ax[1].set_title('final states of two nodes, coloured by class')
plt.show()

The final states are already nicely separated by class: nodes near the resonance frequency have been driven to large amplitudes by class-0 inputs and barely respond to class-1 inputs, while the broadband noise is filtered out. Now we write gradient descent on the readout **by hand**. Note that the network itself is run only once; after that we are just training a linear classifier on the cached final states.

In [ ]:
W_out, b_out = reservoir.params['W_out'], reservoir.params['b_out']     # views into the model: updates are in place
lr, losses, accs = 0.05, [], []
for step in range(300):
    logits = x_T @ W_out.T + b_out                       # forward (readout only)
    loss, dlogits, p = softmax_cross_entropy(logits, y_f) # loss and dL/dlogits = (p - onehot)/N
    gW = dlogits.T @ x_T                                 # dL/dW_out
    gb = dlogits.sum(0)                                  # dL/db_out
    W_out -= lr * gW                                     # gradient descent step
    b_out -= lr * gb
    losses.append(loss); accs.append(accuracy(logits, y_f))

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(losses); ax[0].set_xlabel('gradient step'); ax[0].set_ylabel('cross-entropy loss')
ax[1].plot(accs); ax[1].set_xlabel('gradient step'); ax[1].set_ylabel('training accuracy'); ax[1].set_ylim(0, 1.02)
plt.show()
print(f'test accuracy of the readout-only ("reservoir") HORN: {evaluate(reservoir, S_f_test, y_f_test)[1]:.3f}')

### 3.1 Checking a gradient by brute force

How do we know the formula for the gradient is right? The definition of a derivative gives us a slow but foolproof check: nudge one parameter by a tiny $\epsilon$ in both directions and look at how the loss changes,

$$\frac{\partial L}{\partial \theta} \approx \frac{L(\theta+\epsilon) - L(\theta-\epsilon)}{2\epsilon}.$$

We will use the same trick in Part 4 to validate the full backward pass of the HORN.

In [ ]:
def readout_loss():
    return softmax_cross_entropy(x_T @ W_out.T + b_out, y_f)[0]

_, dlogits, _ = softmax_cross_entropy(x_T @ W_out.T + b_out, y_f)
g_analytic = dlogits.T @ x_T
eps, (i, j) = 1e-5, (1, 3)
W_out[i, j] += eps; L_plus = readout_loss()
W_out[i, j] -= 2 * eps; L_minus = readout_loss()
W_out[i, j] += eps                                   # restore
print(f'dL/dW_out[{i},{j}]   numerical: {(L_plus - L_minus) / (2 * eps):.8f}   analytic: {g_analytic[i, j]:.8f}')

**Try it:**
* Re-run the task with `random_phase=True` in `make_frequency_task`. The accuracy collapses -- why? (Hint: $x_T$ of an oscillator depends on the *phase* of the input; a linear readout of $x_T$ cannot see the amplitude independently of phase. Reading out $|x|$ or $(x, y)$ over a window would help.)
* Compare a homogeneous reservoir with the heterogeneous one, or use a single node (`HORN(1, 1, 2)`).

## 4. Learning II: backpropagation through time

Training only the readout leaves most of the network's potential unused. We also want to learn $W_{ih}, W_{hh}, \mathbf v$ and the biases -- but the loss depends on them *through the whole simulation*: $W_{hh}$ is used at every one of the $T$ time steps. **Backpropagation through time (BPTT)** is nothing more than the chain rule applied to the unrolled sequence of update equations.

The key quantities are the **gradients with respect to the states**, $\mathbf g^x_t = \partial L/\partial \mathbf x_t$ and $\mathbf g^y_t = \partial L/\partial \mathbf y_t$ ("how would the loss change if the state at time $t$ were nudged?"). At the last step we know them from the readout, $\mathbf g^x_T = W_{out}^\top(\mathbf p - \text{onehot})$, $\mathbf g^y_T = 0$. Then we walk **backwards** through time and read each forward equation from right to left, multiplying by its local derivative:

| forward (time $t \to t+1$) | backward (time $t+1 \to t$) |
|---|---|
| $\mathbf x_{t+1} = \mathbf x_t + \mathbf y_{t+1}$ | $\mathbf g^y_{t+1} \mathrel{+}= \mathbf g^x_{t+1}$, $\quad \mathbf g^x_t \mathrel{+}= \mathbf g^x_{t+1}$ |
| $\mathbf y_{t+1} = (1-2\gamma)\mathbf y_t - \omega^2 \mathbf x_t + \alpha\tanh(\mathbf u_t)$ | $\mathbf g^y_t \mathrel{+}= (1-2\gamma)\mathbf g^y_{t+1}$, $\quad \mathbf g^x_t \mathrel{+}= -\omega^2 \mathbf g^y_{t+1}$, $\quad \mathbf g^u_t = \alpha\,(1-\tanh^2\mathbf u_t)\odot \mathbf g^y_{t+1}$ |
| $\mathbf u_t = \tfrac1{\sqrt n}(W_{hh}\mathbf y_t + \mathbf b_{hh} + \mathbf v\odot\mathbf x_t) + W_{ih}\mathbf s_t + \mathbf b_{ih}$ | $\mathbf g^y_t \mathrel{+}= \tfrac1{\sqrt n}W_{hh}^\top \mathbf g^u_t$, $\quad \mathbf g^x_t \mathrel{+}= \tfrac1{\sqrt n}\mathbf v\odot\mathbf g^u_t$ |
| | $\partial L/\partial W_{hh} \mathrel{+}= \tfrac1{\sqrt n}\mathbf g^u_t\,\mathbf y_t^\top$, $\quad \partial L/\partial W_{ih} \mathrel{+}= \mathbf g^u_t\,\mathbf s_t^\top$, $\quad \partial L/\partial\mathbf v \mathrel{+}= \tfrac1{\sqrt n}\mathbf g^u_t\odot\mathbf x_t$, biases $\mathrel{+}= \mathbf g^u_t$ |

Each weight gradient is a *sum over all time steps* of (error signal at that step) $\times$ (what the weight was multiplying at that step) -- a Hebbian-looking product of a post-synaptic error and a pre-synaptic activity. Notice also that the state gradients are propagated by the **transpose of the same linear map $M$** from Part 1: the error signal rings backwards through time with the same frequency and the same slow decay $\approx(1-\gamma)$ per step as the forward dynamics.

This is literally what `HORN.backward` does. Have a look -- it is about 30 lines:

In [ ]:
from IPython.display import Code
Code(inspect.getsource(HORN.backward), language='python')

### 4.1 Gradient check

Before trusting a hand-written backward pass, compare it with finite differences for a few randomly chosen entries of every parameter. Relative errors around $10^{-7}$ or smaller mean the code is correct (we use double precision, so the finite-difference estimate itself is accurate to roughly that level).

In [ ]:
rng = np.random.default_rng(0)
small = HORN.heterogeneous(n_in=2, n_hidden=5, n_out=3, seed=0)
S_chk, y_chk = rng.normal(size=(4, 20, 2)), rng.integers(0, 3, size=4)
errors = check_gradients(small, S_chk, y_chk, n_checks=3, eps=1e-6)
print('\nworst relative error per parameter:', {k: f'{v:.1e}' for k, v in errors.items()})

### 4.2 How far back can the error signal see? Gradient flow through time

A classic problem of recurrent networks is that gradients **vanish** (or explode) as they are propagated backwards over many steps: every step multiplies the error signal by the Jacobian of the update, and products of many such matrices shrink or blow up exponentially. For a plain tanh RNN this limits how far back in time learning can reach.

In a HORN the state Jacobian is the oscillator matrix $M$ (plus the small recurrent term), whose eigenvalues have modulus $\sqrt{1-2\gamma}$. The error signal therefore decays only like $(1-\gamma)^{\Delta t}$ -- a slow, *controlled* fade (the paper calls this a temporal residual connection that stabilises gradients). Let's measure $\|\partial L/\partial\mathbf x_t\|$ for every $t$ in three networks of identical size: an oscillatory HORN, the same HORN with overdamped (integrator-like) nodes, and an Elman tanh RNN.

In [ ]:
T, n = 300, 16
S_g, y_g = rng.normal(size=(8, T, 1)), rng.integers(0, 10, size=8)
models = {f'HORN, gamma = {GAMMA} (oscillatory)': HORN(1, n, 10, seed=0),
          'HORN, gamma = 0.5 (overdamped)': HORN(1, n, 10, gamma=0.5, seed=0),
          'Elman tanh RNN': ElmanRNN(1, n, 10, seed=0)}
fig, ax = plt.subplots(figsize=(9, 3.5))
for name, m in models.items():
    ax.semilogy(gradient_flow(m, S_g, y_g), label=name)
flow = gradient_flow(models[f'HORN, gamma = {GAMMA} (oscillatory)'], S_g, y_g)
ax.semilogy(flow[-1] * (1 - GAMMA) ** (T - np.arange(T + 1)), 'k--', lw=0.8, label=r'$(1-\gamma)^{T-t}$ reference')
ax.set_xlabel('time step t'); ax.set_ylabel(r'$\|\partial L / \partial x_t\|$'); ax.set_ylim(1e-12, None); ax.legend(fontsize=8)
ax.set_title('how the error signal from the readout (at t = T) propagates backwards in time')
plt.show()

The oscillatory HORN keeps a usable error signal over hundreds of steps (with a ripple at the oscillation frequency: the gradient is modulated by the oscillations), the overdamped HORN forgets faster, and the Elman RNN's gradient has vanished after a few dozen steps. Keep this picture in mind for sequential MNIST, where the relevant input is 784 steps long.

**Try it:** vary $\gamma$. How does the memory horizon of learning change?

In [ ]:
def show_flow(gamma=GAMMA):
    m = HORN(1, n, 10, gamma=gamma, seed=0)
    f = gradient_flow(m, S_g, y_g)
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.semilogy(f, label=f'HORN, gamma = {gamma:.3f} ({eigen_analysis(OMEGA, gamma)["regime"]})')
    ax.semilogy(f[-1] * (1 - gamma) ** (T - np.arange(T + 1)), 'k--', lw=0.8, label=r'$(1-\gamma)^{T-t}$')
    ax.set_ylim(1e-12, 10); ax.set_xlabel('time step t'); ax.set_ylabel(r'$\|\partial L/\partial x_t\|$'); ax.legend(); plt.show()

interactive(show_flow, gamma=W.FloatLogSlider(value=GAMMA, base=10, min=-3, max=0, step=0.1, description='gamma'))

## 5. A tiny HORN learns XOR

XOR is the classic problem that a single linear unit cannot solve: the two classes $\{(0,0),(1,1)\}$ and $\{(0,1),(1,0)\}$ are not linearly separable, so some nonlinearity in between input and output is needed. In a HORN we present the two bits as two constant input channels for $T$ steps and read out the answer at the end.

The training loop below is the complete recipe that we will use for everything else: **forward → loss → backward → parameter update**. Instead of plain gradient descent we use **Adam**, which rescales the step for each parameter by a running estimate of its gradient magnitude (see `horn/optim.py`); the paper uses AdamW as well.

In [ ]:
S_xor, y_xor = make_xor(T=30)              # 4 patterns, 30 time steps, 2 input channels
print('input shape (patterns, T, channels):', S_xor.shape, '   labels:', y_xor)
fig, ax = plt.subplots(1, 4, figsize=(12, 1.8), sharey=True)
for i in range(4):
    ax[i].plot(S_xor[i, :, 0], label='bit 1'); ax[i].plot(S_xor[i, :, 1], '--', label='bit 2')
    ax[i].set_title(f'input {xor_bits(i)}  ->  XOR = {y_xor[i]}', fontsize=9); ax[i].set_xlabel('t')
ax[0].legend(fontsize=7); plt.show()

In [ ]:
net_xor = HORN(n_in=2, n_hidden=4, n_out=2, seed=0)
opt = Adam(net_xor.trainable_params(), lr=1e-2)
losses = []
for step in range(400):
    logits, cache = net_xor.forward(S_xor)                        # 1. forward pass (stores states for backward)
    loss, dlogits, _ = softmax_cross_entropy(logits, y_xor)       # 2. loss and its gradient w.r.t. the logits
    grads = net_xor.backward(cache, dlogits)                      # 3. backpropagation through time
    opt.step(grads)                                               # 4. update all trainable parameters
    losses.append(loss)

plt.figure(figsize=(6, 2.8)); plt.semilogy(losses); plt.xlabel('training step'); plt.ylabel('loss'); plt.show()
logits, cache = net_xor.forward(S_xor)
for i in range(4):
    print(f'input {xor_bits(i)}: P(XOR=1) = {softmax(logits)[i, 1]:.3f}   target {int(y_xor[i])}')

Let's look at *how* the network solves the task: the trajectories of the four nodes for the four input patterns, and the readout's "opinion" at every time step (we apply the final readout to the intermediate states $\mathbf x_t$). The decision is not made at the very end -- it builds up during the oscillatory transient.

In [ ]:
X_all = cache['X']                                               # (T+1, 4 patterns, n)
logit_t = X_all @ net_xor.params['W_out'].T + net_xor.params['b_out']   # readout applied at every time step
fig, ax = plt.subplots(1, 5, figsize=(14, 2.8))
for i in range(4):
    plot_traces(X_all[:, i, :], ax=ax[i], title=f'input {xor_bits(i)}, XOR = {y_xor[i]}', xlabel='t', ylabel='x' if i == 0 else '')
    ax[4].plot(logit_t[:, i, 1] - logit_t[:, i, 0], label=f'{xor_bits(i)}')
ax[4].axhline(0, color='k', lw=0.5); ax[4].set_title('readout: logit(1) - logit(0) over time'); ax[4].set_xlabel('t'); ax[4].legend(fontsize=7)
plt.tight_layout(); plt.show()

**Try it:** how small can the network be? Train with 1, 2, 4 hidden nodes, with and without the feedback terms ($\mathbf v$ and the diagonal of $W_{hh}$).

A single node *without* feedback cannot solve XOR: its final state is a monotonic function of its (linear) drive, so the readout is effectively a linear classifier and gets stuck at 3 of 4 patterns (75%). *With* feedback, the same single node usually succeeds -- its response to a strong drive overshoots and swings back by the readout time while a weaker drive is still near its peak. Timing within the oscillation does the nonlinear work. Have a look at the trajectories to see it happen.

In [ ]:
def train_xor(n_hidden=1, feedback=True, seed=0, steps=600):
    m = HORN(2, n_hidden, 2, seed=seed, use_v=feedback)
    if not feedback:
        m.mask_hh = 1.0 - np.eye(n_hidden)            # remove self-connections (velocity feedback)
    hist = train(m, S_xor, y_xor, epochs=steps, batch_size=4, lr=1e-2, verbose=False)
    logits, c = m.forward(S_xor)
    fig, ax = plt.subplots(1, 2, figsize=(11, 2.8))
    ax[0].semilogy(hist['loss']); ax[0].set_xlabel('step'); ax[0].set_ylabel('loss')
    for i in range(4):
        ax[1].plot(c['X'][:, i, 0], label=f'{xor_bits(i)} -> {y_xor[i]}')
    ax[1].axvline(30, color='gray', ls=':'); ax[1].set_title('trajectory of node 0 for the four patterns (readout at t = 30)'); ax[1].legend(fontsize=7)
    plt.show()
    print(f'{n_hidden} node(s), feedback={feedback}: accuracy {accuracy(logits, y_xor):.2f}, params {m.n_params}')

interactive(train_xor, n_hidden=W.IntSlider(value=1, min=1, max=8, description='nodes'),
            feedback=W.Checkbox(value=False, description='feedback v, w'), seed=W.IntSlider(value=0, min=0, max=5))

### 5.1 Temporal XOR: memory and interference

Now a version that needs *memory*: the two bits arrive one after the other on a **single** channel (a pulse for a 1, nothing for a 0), and the answer is read out later. The network must hold the first bit until the second arrives and combine the two nonlinearly.

For oscillators this is natural. The first pulse starts a ringing; the second pulse adds a second ringing that **interferes** with the first. If the pulses are half a period apart, two pulses cancel (destructive interference) -- so the (1,1) pattern ends up looking like (0,0), while a single pulse keeps ringing. Watch a single node:

In [ ]:
def show_interference(delay=14):
    T, t1 = 80, 10
    fig, ax = plt.subplots(figsize=(9, 3))
    for bits, style in [((1, 0), 'C0'), ((0, 1), 'C1--'), ((1, 1), 'C3')]:
        s = bits[0] * impulse(T, t1, 1, 5.0) + bits[1] * impulse(T, t1 + delay, 1, 5.0)
        x, _ = simulate_dho(s, OMEGA, GAMMA, ALPHA)
        ax.plot(x, style, label=f'pulses {bits}')
    ax.axvline(t1, color='gray', lw=0.5); ax.axvline(t1 + delay, color='gray', lw=0.5)
    ax.set_title(f'single node, two pulses {delay} steps apart ({delay / (2*np.pi/OMEGA):.2f} periods)'); ax.set_xlabel('time step'); ax.legend(); plt.show()

interactive(show_interference, delay=W.IntSlider(value=14, min=1, max=40, description='delay'))

In [ ]:
S_txor, y_txor = make_temporal_xor(T=40, t1=5, t2=19)        # pulses 14 steps = half a period apart, readout at T = 40
fig, ax = plt.subplots(1, 4, figsize=(12, 1.8), sharey=True)
for i in range(4):
    ax[i].plot(S_txor[i, :, 0]); ax[i].set_title(f'bits {xor_bits(i)} -> XOR = {y_txor[i]}', fontsize=9); ax[i].set_xlabel('t')
plt.show()

net_t = HORN(n_in=1, n_hidden=4, n_out=2, seed=0)
hist_t = train(net_t, S_txor, y_txor, epochs=600, batch_size=4, lr=1e-2, verbose=False)
logits, cache = net_t.forward(S_txor)
print('accuracy:', accuracy(logits, y_txor), '  P(XOR=1) per pattern:', softmax(logits)[:, 1].round(3))

X_all = cache['X']; logit_t = X_all @ net_t.params['W_out'].T + net_t.params['b_out']
fig, ax = plt.subplots(1, 5, figsize=(14, 2.8))
for i in range(4):
    plot_traces(X_all[:, i, :], ax=ax[i], title=f'bits {xor_bits(i)}, XOR = {y_txor[i]}', xlabel='t', ylabel='x' if i == 0 else '')
    ax[4].plot(logit_t[:, i, 1] - logit_t[:, i, 0], label=f'{xor_bits(i)}')
ax[4].axhline(0, color='k', lw=0.5); ax[4].set_title('readout over time'); ax[4].set_xlabel('t'); ax[4].legend(fontsize=7)
plt.tight_layout(); plt.show()

**Try it:** change the spacing of the pulses (`t2`) to a quarter period (`t2 = 12`) or a full period (`t2 = 33`), and the readout time `T`. In our experiments the half-period spacing was the most reliably learnable with 4 nodes; with other spacings more nodes or more training steps are often needed. Try also a heterogeneous network (`HORN.heterogeneous`) and different seeds.

## 6. Sequential MNIST

The paper's main benchmark is **sequential MNIST (sMNIST)**: each $28\times28$ digit is read out pixel by pixel in scan-line order and fed to the network as a time series of length $784$ with a *single* input channel. The network sees one pixel per time step and has to name the digit at the end. For a recurrent network this is hard: the relevant information is spread over hundreds of steps.

For oscillators, however, the scan-line format has a wonderful property. A **vertical stroke** in the image produces a bright pixel every 28 steps -- a periodic signal with frequency $\omega_v = 2\pi/28$, which is exactly the default natural frequency of the nodes. Strokes of other slopes produce nearby frequencies. The geometry of the digit is converted into a *spectral* pattern that resonating nodes can pick up directly ("feature extraction through resonance", in the words of the paper).

In [ ]:
tr_img, tr_lbl, te_img, te_lbl = load_mnist()            # downloads ~11 MB once (cached in data/mnist.npz)
S_tr, y_tr = to_sequences(tr_img), tr_lbl.astype(int)     # (60000, 784, 1) float32 in [0, 1]
S_te, y_te = to_sequences(te_img), te_lbl.astype(int)
perm = np.random.default_rng(0).permutation(len(S_tr)); val_idx, tr_idx = perm[:1000], perm[1000:]   # hold out 1000 for validation
S_val, y_val = S_tr[val_idx], y_tr[val_idx]
print('train', S_tr.shape, 'test', S_te.shape)

i = 7
fig = plt.figure(figsize=(12, 3)); gs = fig.add_gridspec(1, 4)
ax0 = fig.add_subplot(gs[0, 0]); ax0.imshow(tr_img[i], cmap='gray_r'); ax0.set_title(f'digit {y_tr[i]}'); ax0.axis('off')
ax1 = fig.add_subplot(gs[0, 1:]); ax1.plot(S_tr[i, :, 0], lw=0.8)
for r in range(1, 28): ax1.axvline(28 * r, color='gray', lw=0.3)
ax1.set_title('the same digit as a 784-step time series (grey lines: row boundaries every 28 steps)'); ax1.set_xlabel('time step (pixel index)')
plt.show()

In [ ]:
# average power spectrum of sMNIST time series: peaks at omega_v = 2 pi / 28 and its harmonics
P = np.mean([power_spectrum(S_tr[j, :, 0])[1] for j in range(500)], axis=0)
f, _ = power_spectrum(S_tr[0, :, 0])
fig, ax = plt.subplots(1, 2, figsize=(12, 3))
ax[0].semilogy(f[1:], P[1:]); [ax[0].axvline(k * 2 * np.pi / 28, color='C3', ls=':', lw=0.8) for k in range(1, 6)]
ax[0].set_xlabel('frequency (rad / step)'); ax[0].set_ylabel('mean power'); ax[0].set_title(r'spectrum of sMNIST digits (red: multiples of $\omega_v = 2\pi/28$)')

# a single node tuned to omega_v resonates with vertical strokes
for name, ls in [('vertical', '-'), ('diagonal', '--'), ('horizontal', ':')]:
    s = bar_image(name).reshape(-1)
    x, _ = simulate_dho(s, OMEGA, GAMMA, ALPHA)
    ax[1].plot(x, ls, label=f'{name} bar')
ax[1].set_title(r'response of one node ($\omega = \omega_v$) to bar images read as time series'); ax[1].set_xlabel('time step'); ax[1].legend(); plt.show()

### 6.1 Training a HORN on sMNIST

We use the authors' default configuration: 32 homogeneous nodes with $\omega = 2\pi/28$, $\gamma = 0.01$, $\alpha = 0.04$, Adam(W) with learning rate $10^{-2}$, mini-batches of 64 digits. Each training step is a forward and a BPTT backward pass through 784 time steps -- in NumPy about 0.17 s. We first train for a limited number of steps right here so you can watch learning happen (about a minute), then load models that were trained for two full epochs with `scripts/train_smnist.py` (about 5 minutes each on a CPU).

In [ ]:
QUICK_STEPS = 150
net_quick = HORN(n_in=1, n_hidden=32, n_out=10, seed=0)
print('trainable parameters:', net_quick.n_params)
hist_quick = train(net_quick, S_tr[tr_idx], y_tr[tr_idx], epochs=1, batch_size=64, lr=1e-2, weight_decay=1e-2,
                   S_val=S_val, y_val=y_val, eval_every=25, max_steps=QUICK_STEPS)
plot_history(hist_quick, k=10); plt.show()

In [ ]:
PRE = REPO / 'pretrained'

def load_pretrained(name):
    '''trained model, its untrained twin, and the training history (or Nones if not available)'''
    if not (PRE / f'{name}.npz').exists():
        return None, None, None
    cls = ElmanRNN if name.startswith('elman') else HORN
    hist = {k: v for k, v in np.load(PRE / f'{name}_history.npz').items()}
    return cls.load(PRE / f'{name}.npz'), cls.load(PRE / f'{name}_init.npz'), hist

runs = {'homogeneous HORN': 'horn_32', 'heterogeneous HORN': 'horn_hetero_32',
        'overdamped HORN (gamma = 0.5)': 'horn_overdamped_32', 'Elman tanh RNN': 'elman_32'}
pre = {label: load_pretrained(name) for label, name in runs.items()}
pre = {k: v for k, v in pre.items() if v[0] is not None}

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for label, (m, m0, h) in pre.items():
    plot_history(h, ax=ax, label=f"{label} (test acc {float(h['test_acc']):.3f})", k=30, steps_per_epoch=int(np.ceil(59000 / 64)))
ax[1].legend(fontsize=8); ax[0].set_title('2 epochs of sMNIST training, ~1,450-1,480 parameters each'); plt.show()
if not pre:
    print('no pretrained models found -- run scripts/train_smnist.py (see README) or use net_quick below')

Compare with Fig. 1B of the paper: at this small size the oscillatory HORNs learn fast, the heterogeneous network is slightly ahead, the *overdamped* network with otherwise identical architecture learns much more slowly, and the Elman RNN with the same number of parameters does not learn at all within two epochs (its gradient cannot reach back through 784 steps). Note that two epochs is very short: the paper trains for 100 epochs and reaches well above 95% with networks of this size.

From here on we analyse the trained homogeneous and heterogeneous HORNs (falling back to `net_quick` if the pretrained files are missing).

In [ ]:
model_h, init_h, hist_h = pre.get('homogeneous HORN', (net_quick, HORN(1, 32, 10, seed=0), hist_quick))
model_n, init_n, hist_n = pre.get('heterogeneous HORN', (None, None, None))

N_TEST = 2000
pred = model_h.predict(S_te[:N_TEST])
print(f'homogeneous HORN: accuracy on {N_TEST} test digits = {np.mean(pred == y_te[:N_TEST]):.3f}')
C = confusion_matrix(y_te[:N_TEST], pred)
fig, ax = plt.subplots(figsize=(4.2, 3.8))
im = ax.imshow(C, cmap='Blues'); plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_xticks(range(10)); ax.set_yticks(range(10)); ax.grid(False); ax.set_title('confusion matrix'); plt.show()

### 6.2 Looking inside the trained network

The paper's Fig. 2 describes how learning reshapes the network dynamics. We can reproduce its main panels on our miniature models.

**Activity patterns (Fig. 2A).** Before training, the homogeneous network responds to a digit with globally synchronised waves. After training the responses are irregular and stimulus specific.

In [ ]:
digit = np.where(y_te == 0)[0][0]
S_digit = S_te[digit:digit + 1]
panels = [('homogeneous, untrained', init_h), ('homogeneous, trained', model_h)]
if model_n is not None:
    panels += [('heterogeneous, untrained', init_n), ('heterogeneous, trained', model_n)]
fig, axes = plt.subplots(len(panels) // 2, 2, figsize=(13, 2.6 * len(panels) // 2), sharex=True, squeeze=False)
for ax, (name, m) in zip(axes.ravel(), panels):
    plot_traces(m.simulate(S_digit)['X'][0], ax=ax, title=f'{name}: response to a "0"', lw=0.7, xlabel='time step')
plt.tight_layout(); plt.show()

**Decorrelation (Fig. 2B, C).** Pairwise cross-correlations between node activities are close to $\pm 1$ in the untrained homogeneous network (everything is synchronised) and become much more diverse after training: learning *desynchronises* the network and thereby increases the dimensionality of its activity. The Kuramoto order parameter tells the same story in time (Fig. 2D).

In [ ]:
n_per_class = 10
idx = np.concatenate([np.where(y_te == c)[0][:n_per_class] for c in range(10)])
X_init = init_h.simulate(S_te[idx])['X'][:, 1:]          # (100, 784, n)
X_trained = model_h.simulate(S_te[idx])['X'][:, 1:]

fig, ax = plt.subplots(1, 4, figsize=(14, 3.2))
for j, (name, X) in enumerate([('untrained', X_init), ('trained', X_trained)]):
    CC = pairwise_crosscorr(X)
    plot_matrix(CC, ax=ax[2 * j], title=f'cross-correlations, {name}', colorbar=False)
    ax[2 * j + 1].hist(upper_triangle(CC), bins=np.linspace(-1, 1, 41), color=f'C{j}')
    ax[2 * j + 1].set_title(f'distribution, {name}'); ax[2 * j + 1].set_xlabel('correlation coefficient')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 3), sharey=True)
for a, (name, X) in zip(ax, [('untrained', X_init), ('trained', X_trained)]):
    r = kuramoto_order(X)                                 # (100, 784)
    for c in range(10):
        a.plot(r[y_te[idx] == c].mean(0), lw=0.8, label=str(c))
    a.set_title(f'Kuramoto order parameter r(t), mean per digit class -- {name}'); a.set_xlabel('time step'); a.set_ylim(0, 1)
ax[0].legend(fontsize=7, ncol=5, title='digit'); plt.show()

**Intrinsic receptive fields (Fig. 2E).** Learning changes the diagonal of $W_{hh}$ (velocity feedback $w$) and $\mathbf v$ (amplitude feedback) of every node, and with them the node's gain curve. In the homogeneous network all nodes start with the same gain curve and spread out during learning; the heterogeneous network covers the frequency axis from the start.

In [ ]:
freqs_irf = np.linspace(0.02, 1.0, 300)
panels = [('homogeneous, untrained', init_h), ('homogeneous, trained', model_h)]
if model_n is not None:
    panels += [('heterogeneous, untrained', init_n), ('heterogeneous, trained', model_n)]
fig, axes = plt.subplots(1, len(panels), figsize=(3.3 * len(panels), 3), sharey=True)
for ax, (name, m) in zip(np.atleast_1d(axes), panels):
    G = intrinsic_receptive_fields(m, freqs_irf)
    colors = plt.get_cmap('viridis')(np.linspace(0, 1, m.n_hidden))
    for i in range(m.n_hidden):
        ax.semilogy(freqs_irf, G[i], color=colors[i], lw=0.7)
    ax.axvline(2 * np.pi / 28, color='k', ls=':', lw=0.8); ax.axhline(1, color='gray', lw=0.5)
    ax.set_title(name, fontsize=9); ax.set_xlabel(r'input frequency $\omega_i$'); ax.set_ylim(1e-2, 1e3)
np.atleast_1d(axes)[0].set_ylabel('gain G'); plt.tight_layout(); plt.show()
w_h = np.diag(model_h.params['W_hh']) * model_h.gain_rec
print(f'learned velocity feedback (scaled), homogeneous net: min {w_h.min():.3f}, max {w_h.max():.3f}  (self-sustained for w > {2*GAMMA/ALPHA*1:.2f})')

**Hebbian-like weight changes (Fig. 4A).** The paper notices that BPTT produces weight changes compatible with Hebb's rule: after training, the weight $W_{hh,ij}$ from node $j$ to node $i$ correlates with the correlation of their activities. Let's check.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for a, (name, m, X) in zip(ax, [('untrained', init_h, X_init), ('trained', model_h, X_trained)]):
    CC, Wm = pairwise_crosscorr(X), m.params['W_hh']
    off = ~np.eye(m.n_hidden, dtype=bool)
    a.scatter(CC[off], Wm[off], s=6, alpha=0.5)
    rho = np.corrcoef(CC[off], Wm[off])[0, 1]
    k, d = np.polyfit(CC[off], Wm[off], 1); xx = np.linspace(-1, 1, 10); a.plot(xx, k * xx + d, 'r', lw=1)
    a.set_title(f'{name}: corr(CC_ij, W_ij) = {rho:.2f}'); a.set_xlabel('activity correlation CC_ij'); a.set_ylabel('recurrent weight W_hh,ij')
plt.tight_layout(); plt.show()

**Evidence accumulation.** When is the decision made? We apply the trained readout to the intermediate states $\mathbf x_t$ and measure the accuracy as a function of time. The readout was trained for $t = 784$ only, so this is a lower bound on how much information is present earlier (the paper trains a separate decoder for each time point, SI Fig. S2).

In [ ]:
N_ACC = 1000
out = model_h.simulate(S_te[:N_ACC])
logit_t = out['X'] @ model_h.params['W_out'].T + model_h.params['b_out']          # (N, T+1, 10)
acc_t = (logit_t.argmax(-1) == y_te[:N_ACC, None]).mean(0)
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(acc_t); ax.axhline(0.1, color='gray', ls=':', label='chance'); ax.set_xlabel('time step t (pixel index)'); ax.set_ylabel('accuracy of the final readout applied at t')
ax.set_title('the network accumulates evidence over the digit'); ax.legend(); plt.show()

**Noise robustness (Fig. 1E, F) and lesions (Fig. 2G).** The frequency-selective gain of the nodes suppresses high-frequency noise, so HORNs degrade gracefully when pixel noise is added at test time -- even though our model was trained on clean digits. And because the representation is distributed over all nodes, silencing any single node costs only a little accuracy.

In [ ]:
sigmas = [0.0, 0.1, 0.25, 0.5, 1.0]
accs = [evaluate(model_h, add_noise(S_te[:N_ACC], s, seed=1), y_te[:N_ACC])[1] for s in sigmas]

def lesion(model, i):
    m = model.copy()
    for k in ('W_ih', 'b_ih', 'b_hh'):
        m.params[k][i] = 0
    m.params['W_hh'][i, :] = 0; m.params['W_hh'][:, i] = 0; m.params['W_out'][:, i] = 0
    return m
base = evaluate(model_h, S_te[:500], y_te[:500])[1]
lesion_acc = [evaluate(lesion(model_h, i), S_te[:500], y_te[:500])[1] for i in range(model_h.n_hidden)]

fig, ax = plt.subplots(1, 2, figsize=(12, 3))
ax[0].plot(sigmas, accs, 'o-'); ax[0].set_xlabel(r'test noise $\sigma$ (pixel values are in [0, 1])'); ax[0].set_ylabel('test accuracy'); ax[0].set_ylim(0, 1); ax[0].set_title('robustness to additive pixel noise')
ax[1].bar(range(model_h.n_hidden), lesion_acc); ax[1].axhline(base, color='k', ls='--', label='intact network'); ax[1].set_xlabel('silenced node'); ax[1].set_ylabel('accuracy'); ax[1].set_ylim(0, 1); ax[1].legend(); ax[1].set_title('single-node lesions')
plt.tight_layout(); plt.show()

### 6.3 Reservoir computing on sMNIST

Finally, back to Part 3: how far do we get **without** training the recurrent network at all? We run the untrained homogeneous and heterogeneous HORNs over 5,000 digits once, cache the final states, and fit only the readout. This is the "reservoir" control of the paper's Fig. 4B. Heterogeneity pays off immediately: the synchronised homogeneous reservoir carries little information in its final state, the heterogeneous one considerably more -- but both are far from the trained network.

In [ ]:
N_RES = 5000
feats = {}
for name, m in [('homogeneous', HORN(1, 32, 10, seed=0)), ('heterogeneous', HORN.heterogeneous(1, 32, 10, seed=0))]:
    xT_tr = np.concatenate([m.forward(S_tr[tr_idx[i:i + 500]])[1]['X'][-1] for i in range(0, N_RES, 500)])
    xT_te = np.concatenate([m.forward(S_te[i:i + 500])[1]['X'][-1] for i in range(0, N_TEST, 500)])
    W_o, b_o = np.zeros((10, 32)), np.zeros(10)
    opt = Adam({'W': W_o, 'b': b_o}, lr=1e-2)
    for step in range(500):
        _, dl, _ = softmax_cross_entropy(xT_tr @ W_o.T + b_o, y_tr[tr_idx[:N_RES]])
        opt.step({'W': dl.T @ xT_tr, 'b': dl.sum(0)})
    acc_tr = accuracy(xT_tr @ W_o.T + b_o, y_tr[tr_idx[:N_RES]]); acc_te = accuracy(xT_te @ W_o.T + b_o, y_te[:N_TEST])
    print(f'{name:14s} reservoir (readout only): train acc {acc_tr:.3f}, test acc {acc_te:.3f}')

## 7. Where to go from here

You now have every ingredient of the paper in a form you can modify. Some suggestions, roughly in order of effort:

1. **Non-oscillatory control.** Train `HORN(1, 32, 10, gamma=0.5)` yourself with `scripts/train_smnist.py --gamma 0.5` and compare to `gamma=0.01`; or sweep $\omega$ (the paper's Fig. 1D): what happens with $\omega$ far from $2\pi/28$?
2. **Permuted sMNIST** (`to_sequences(images, permutation=...)`, or `--permute` in the script) destroys the spectral structure. Does heterogeneity help more here, as in the paper?
3. **Noisy training** (`add_noise`) and the robustness curve of Fig. 1F.
4. **Two-layer HORNs.** `HORN.mask_hh` is a binary mask on $W_{hh}$ that is applied in the forward and backward pass. Build a network of $n_1 + n_2$ nodes with sparse feed-forward and feedback connections between the two blocks, give layer 2 a lower $\omega$ and $\gamma$ (the paper's Fig. 3B), feed the input only to layer 1 and read out only from layer 2 (zero and freeze the corresponding rows of `W_ih` / columns of `W_out`).
5. **Conduction delays.** Store the last $d_{max}$ velocity vectors and let each connection read from its own delay (paper Fig. 3A). This needs a change to `forward` *and* `backward` -- a good exercise in BPTT.
6. **Hebbian learning.** Replace the BPTT update of $W_{hh}$ by $\Delta W_{ij} = \pm\lambda\, r(x_i, x_j)$ with the activity correlations from `pairwise_crosscorr`, while input and readout keep learning with BPTT (paper Fig. 4B). Does the anti-Hebbian rule desynchronise the homogeneous network as the paper reports?
7. **Geometric input and travelling waves.** Give each of $14\times14$ nodes a fixed $2\times2$ pixel receptive field, flash the digit for one time step and read out after 150 steps of free dynamics (paper Fig. 5). Watch for Mexican-hat connectivity after training.

The reference implementation ([exilef/horn](https://github.com/exilef/horn)) is a PyTorch module of about 100 lines; everything you learned here transfers directly, with autograd taking over the role of `HORN.backward`.